In [1]:
#include <Wire.h>
#include <LiquidCrystal_I2C.h>

// LCD Setup (Assuming 0x27 I2C address)
LiquidCrystal_I2C lcd(0x27, 16, 2);

// =========================
// PIN DEFINITIONS & CONSTANTS
// =========================

// Ultrasonic Sensor 1 (Start Sensor - Detects initial object presence)
const int trig1 = 9;
const int echo1 = 8;

// Ultrasonic Sensor 2 (Stop Sensor - Measures time/speed)
const int trig2 = 7;
const int echo2 = 6;

// Buzzer Pin
const int buzzer = 4;

// CONSTANT: Assumed distance the object travels in meters (REQUIRED for speed calc)
const float TRAVEL_DISTANCE_M = 0.3; 

// Speed Limit (m/s)
const float SPEED_LIMIT = 0.06; // Example: ~216 km/h limit if measured over a short distance

// STATE VARIABLES
unsigned long t1_start_time = 0; // Time object enters sensor 1 range
bool object_started = false;   // True when object is detected by Sensor 1
bool tracking_active = false;  // True when measurement phase is active (between sensors)

// DANGER BUZZER CONTROL (NON-BLOCKING)
bool dangerModeActive = false;
unsigned long lastBuzzTime = 0;
int buzzCount = 0;


// LCD Cache
String lastLine1 = "";
String lastLine2 = "";


/**
 * @brief Updates the LCD display only if the content has changed.
 */
void updateLCD(String line1, String line2) {
    if (line1 != lastLine1 || line2 != lastLine2) {
        lcd.setCursor(0, 0);
        // Clear previous contents of Line 1 before printing new data
        lcd.print("                "); // Print spaces to clear up to 16 characters

        lcd.setCursor(0, 0);
        lcd.print(line1);

        lcd.setCursor(0, 1);
        // Clear previous contents of Line 2 before printing new data
        lcd.print("                "); // Print spaces to clear up to 16 characters

        lcd.setCursor(0, 1);
        lcd.print(line2);

        lastLine1 = line1;
        lastLine2 = line2;
    }
}


/**
 * @brief Reads the distance from a single ultrasonic sensor (cm).
 * NOTE: Uses pulseIn for timing, which can be susceptible to loop jitter.
 */
long readDistance(int trig, int echo) {
    // Ensure clean start
    digitalWrite(trig, LOW);
    delayMicroseconds(2);

    // Send burst
    digitalWrite(trig, HIGH);
    delayMicroseconds(10);
    digitalWrite(trig, LOW);

    // Measure duration (Timeout set high enough to prevent freezing)
    long duration = pulseIn(echo, HIGH, 30000UL); 

    // Calculate distance in cm: Speed of sound = 340 m/s or 29 microseconds/cm.
    // Distance = Duration * (Speed of Sound / 2)
    // We divide by 2 because the pulse travels TO and FROM the object.
    long cm = duration * 0.034 / 2;

    return cm;
}


void setup() {
    Serial.begin(9600);
    lcd.init();
    lcd.backlight();

    // Sensor Setup
    pinMode(trig1, OUTPUT);
    pinMode(echo1, INPUT);

    pinMode(trig2, OUTPUT);
    pinMode(echo2, INPUT);

    // Buzzer Setup
    pinMode(buzzer, OUTPUT);

    updateLCD("SMART SPEED", "DETECTOR");
    delay(1500);
}


void loop() {
    // --- 1. READ ALL SENSOR DATA ---
    long d1 = readDistance(trig1, echo1);
    long d2 = readDistance(trig2, echo2);

    String statusLine = "IDLE";
    String speedLine = "";


    // --- 2. START CONDITION (Sensor 1: Object enters field) ---
    if (d1 < 10 && !object_started) {
        t1_start_time = millis(); // Record start time
        object_started = true;   // Activate state machine flag

        updateLCD("OBJECT", "DETECTED");
        tone(buzzer, 1000, 120);
    }


    // --- 3. STOP/MEASUREMENT CONDITION (Sensor 2: Object leaves field) ---
    if (d2 < 10 && object_started && !tracking_active) {
        // We have transitioned from the start phase to the stop/measurement phase
        
        t2 = millis(); // Record end time
        tracking_active = true; // Set active flag

        float timeSec = (float)(t2 - t1_start_time) / 1000.0; // Total time elapsed in seconds
        
        // Handle division by zero or minimal travel time
        if (timeSec < 0.05) {
            statusLine = "TOO FAST?";
            speedLine = "---";
        } else {
            // Calculate speed: Speed = Distance / Time
            float speed = TRAVEL_DISTANCE_M / timeSec;
            
            Serial.print("Speed: ");
            Serial.println(speed);

            if (speed < SPEED_LIMIT) {
                statusLine = "SAFE";
                speedLine = String(speed, 2) + " m/s";
            }
            else if (speed < SPEED_LIMIT * 1.5) {
                statusLine = "WARNING";
                tone(buzzer, 2000, 250); // Warning buzz
                speedLine = String(speed, 2) + " m/s";
            }
            else {
                statusLine = "DANGER";

                // START NON-BLOCKING DANGER MODE (Resets and initiates high alert)
                dangerModeActive = true;
                buzzCount = 0;
                lastBuzzTime = millis();
                speedLine = String(speed, 2) + " m/s";
            }
        }

        updateLCD(statusLine, speedLine);
    }


SyntaxError: invalid decimal literal (382025273.py, line 84)